In [1]:
from dfbr.utils.files import get_config
from dfbr.data.dataset import BikeDemandDataset
from dfbr.models.station_targets import BikeStationTargets
from dfbr.models.cost_head import CostHead
from dfbr.models.mlp import MLP
from dfbr.eval.simmulation import Sim, create_station_dict, create_event_df
from dfbr.training.train import get_loss_func, train_one_epoch, evaluate
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch.nn as nn
import pyepo

#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Setup
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Read config
config = get_config("baseline.yaml")

#Create a dictionary of stations
station_dict = create_station_dict(config["paths"]["stations"], config["paths"]["station_dist_miles"], config["sim"]["start_inv_pct"])
#Sort by id to ensure alignment
station_ids = sorted(station_dict.keys())
#Get parameters for shapes of datasets and models
num_stations = len(station_dict)
capacities = [s.capacity for s in station_dict.values()]
max_cap = max(capacities)




In [2]:
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Load Datasets
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Create datasets
train_ds = BikeDemandDataset(
        file = config["paths"]["input"],
        start_date = config["data"]["train_start_date"],
        end_date = config["data"]["train_end_date"],
        target_cols= [str(id) for id in station_ids],
        input_scale_cols= ['mean_temp', 'precip', 'max_gust'],
        input_no_scale_cols=['sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month'],
        capacities=capacities,
        max_cap=max_cap
    )


#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Solve for optimal values with ground truth data
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
optmodel = BikeStationTargets(num_stations=num_stations, max_cap=max_cap, total_inventory=config["model"]["total_inventory"])
pyepo_train_ds = pyepo.data.dataset.optDataset(optmodel, train_ds.X, train_ds.c.view(-1, num_stations * (max_cap + 1)))

#Wrap Data Loaders
train_dl = DataLoader(train_ds, batch_size=config["training"]["batch_size"], shuffle=False)


Set parameter Username
Set parameter LicenseID to value 2774727
Academic license - for non-commercial use only - expires 2027-02-03
Optimizing for optDataset...


100%|██████████| 731/731 [00:36<00:00, 20.14it/s]


In [4]:
pyepo_train_ds[0]

(tensor([-0.4349, -0.5169,  0.4653, -0.7818,  0.6235,  0.5000,  0.8660]),
 tensor([0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 1.0000e+09, 1.0000e+09,
         1.0000e+09]),
 tensor([0., -0., -0.,  ..., -0., -0., -0.]),
 tensor([0.]))